# 02 · empirical quantile per (state × clock band)

No model and no extrapolation: **look up what actually happened**.

```
state = active | warm | idle           from burn in the last 30 min / 2 h
band  = hour_of_day // width
answer = the q-th percentile of what the human really burned in the next N minutes,
         over every training anchor in the same cell
```

Three knobs, all tuned by cost: the band width, the minimum samples before a cell is
trusted, and **`q` itself** — which the cost model says should land near 0.909.

The weakness is sample size. Anchors sit 5 minutes apart and overlap, so a cell with
`n = 400` holds perhaps 15–20 distinct work sessions and its quantile is set by a
handful of them. The backoff hierarchy is what stops that becoming nonsense.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd() / "lab"))
import numpy as np, pandas as pd
import dataset as D

frame = D.build("claude")
train, validation, test = D.split(frame)
COLS = D.features(frame)
print("\ncost model:  stolen $%.2f/pct   wasted $%.2f/pct   ratio %.0f:1"
      % (D.COST_STOLEN, D.COST_WASTED, D.COST_STOLEN / D.COST_WASTED))
print("the cost-optimal quantile is therefore %.4f" % D.optimal_quantile())

12075 anchors over 41.9 days, embargo 300 min at each seam
  train        7185 rows  08-24 17:55 -> 09-18 16:35
  validation   2355 rows  09-18 21:40 -> 09-27 01:50
  test         2415 rows  09-27 06:55 -> 10-05 16:05
  dropped to the embargo: 120

cost model:  stolen $3.20/pct   wasted $0.32/pct   ratio 10:1
the cost-optimal quantile is therefore 0.9091


## The objective

Not accuracy — **money**. Two unit costs per percent of a 5-hour meter:

| | $/pct | Meaning |
|---|---|---|
| `COST_WASTED` | **0.32** | quota the bot was refused although the human never came — exactly 1% of a $32 unit |
| `COST_STOLEN` | **3.20** | quota the human wanted and the bot took — 10×, because the user has first claim |

Hyper-parameters below are chosen on **validation**, by cost. The test set is read
only in `05_comparison.ipynb`.

In [2]:
def fit_table(fit_frame, N, width, min_n, q):
    y = fit_frame["y%d" % N].values
    s = D.state(fit_frame)
    b = (fit_frame["hour"].values // width) * width
    ok = ~np.isnan(y)
    cell, per_state = {}, {}
    overall = float(np.percentile(y[ok], q * 100))
    for st in D.STATES:
        m = ok & (s == st)
        if m.sum() >= min_n:
            per_state[st] = float(np.percentile(y[m], q * 100))
        for bd in np.unique(b):
            mm = m & (b == bd)
            if mm.sum() >= min_n:
                cell[(st, int(bd))] = float(np.percentile(y[mm], q * 100))
    return cell, per_state, overall, width

def predict_table(model, f):
    cell, per_state, overall, width = model
    s = D.state(f)
    b = (f["hour"].values // width) * width
    return np.array([cell.get((st, int(bd)), per_state.get(st, overall))
                     for st, bd in zip(s, b)])

candidates = {"w=%d n>=%d q=%.3f" % (w, n, q): (w, n, q)
              for w in (2, 3, 4, 6, 12)
              for n in (10, 30, 100)
              for q in (0.75, 0.85, 0.909, 0.95)}
def make(c, N):
    return predict_table(fit_table(train, N, *c), validation)
chosen, table = D.tune("state x clock", candidates, validation, make, show=4)

N = 30   best: w=2 n>=100 q=0.750           $0.4621   (worst $1.3227)
            config  cost_usd  stolen_usd  wasted_usd
w=2 n>=100 q=0.750    0.4621      0.3913      0.0708
w=4 n>=100 q=0.750    0.4691      0.4158      0.0533
w=4 n>=100 q=0.850    0.4696      0.2541      0.2155
w=3 n>=100 q=0.750    0.4911      0.4511      0.0399

N = 60   best: w=6 n>=100 q=0.850           $0.9994   (worst $3.2167)
             config  cost_usd  stolen_usd  wasted_usd
 w=6 n>=100 q=0.850    0.9994      0.4157      0.5837
  w=6 n>=10 q=0.850    1.0023      0.4157      0.5866
 w=2 n>=100 q=0.850    1.0149      0.4582      0.5567
w=12 n>=100 q=0.850    1.0186      0.4905      0.5280

N = 120  best: w=6 n>=10 q=0.850            $2.0862   (worst $8.5914)
            config  cost_usd  stolen_usd  wasted_usd
 w=6 n>=10 q=0.850    2.0862      0.4636      1.6227
 w=6 n>=30 q=0.850    2.1116      0.4636      1.6480
w=6 n>=100 q=0.850    2.2435      0.4636      1.7799
w=3 n>=100 q=0.850    2.2803      0.5867 

In [3]:
# Does the cost-optimal q match the theory (0.909)?
t = table.copy()
t["q"] = [float(s.split("q=")[1]) for s in t["config"]]
print(t.groupby("q")["cost_usd"].min().round(4).to_string())
print("\ntheory says %.3f" % D.optimal_quantile())

q
0.750    0.4621
0.850    0.4696
0.909    0.5910
0.950    0.8637

theory says 0.909


In [4]:
ms = {N: fit_table(train, N, *chosen[N][1]) for N in D.HORIZONS}
val = [D.evaluate("state x clock", validation["y%d" % N].values,
                  predict_table(ms[N], validation)) | {"N": N} for N in D.HORIZONS]
print(pd.DataFrame(val).set_index("N")[["cost_usd", "stolen", "wasted", "coverage"]].round(3))
print()
print(D.by_cell(validation, validation["y300"].values,
                {"state x clock": predict_table(ms[300], validation)}).to_string(index=False))
D.save("02_state_clock", {"kind": "state_clock",
                          "per_horizon": {str(N): {"band_hours": chosen[N][1][0],
                                                   "min_n": chosen[N][1][1],
                                                   "q": chosen[N][1][2]}
                                          for N in D.HORIZONS}}, val)

     cost_usd  stolen  wasted  coverage
N                                      
30      0.462   0.122   0.221    95.287
60      0.999   0.130   1.824    94.480
120     2.086   0.145   5.071    92.909
180     3.229   0.212   7.969    90.403
300     6.664   1.085   9.977    85.902

 state      band   n  ACTUAL p95  state x clock
  idle     night 759         5.0            0.0
  idle   morning 421        38.0           16.1
  idle afternoon 253        40.0           28.1
  idle   evening 479         0.0            1.5
  warm     night  20         3.0           39.0
  warm   morning  28        83.0           25.8
  warm afternoon  62        84.9           64.6
  warm   evening  92         0.0           54.5
active     night  12         3.0           54.0
active   morning  31        79.0           75.3
active afternoon 165        84.0           81.1
active   evening  33         5.4           57.4
froze 02_state_clock -> design/02_prediction/lab/results/02_state_clock.json


PosixPath('/Users/jeanlescut/dev/agent-quota-maximizer/design/02_prediction/lab/results/02_state_clock.json')